In [0]:
gold_turnout_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/election_turnout/state/"
)
gold_state_turnout=spark.read.parquet(gold_turnout_path)

In [0]:
gold_state_turnout.groupBy(
    "bundesland",
    "wahljahr",
    "gebietstyp"
).count().orderBy(
    "bundesland",
    "wahljahr",
    "gebietstyp"
).show(200, truncate=False)

+----------------------+--------+---------------+-----+
|bundesland            |wahljahr|gebietstyp     |count|
+----------------------+--------+---------------+-----+
|Bayern                |2023    |11             |507  |
|Bayern                |2023    |21             |521  |
|Hamburg               |2020    |Wahlbezirk     |1884 |
|Hamburg               |2025    |BRIEFWAHLBEZIRK|703  |
|Hamburg               |2025    |STIMMBEZIRK    |1269 |
|Mecklenburg-Vorpommern|2021    |Wahlbezirk     |2003 |
|Nordrhein-Westfalen   |2017    |Wahlkreis      |129  |
|Nordrhein-Westfalen   |2022    |Wahlkreis      |129  |
|Sachsen               |2019    |BW             |739  |
|Sachsen               |2019    |UB             |3579 |
|Sachsen               |2024    |BW             |1100 |
|Sachsen               |2024    |UB             |3440 |
|Sachsen-Anhalt        |2021    |B              |429  |
|Sachsen-Anhalt        |2021    |U              |2199 |
|Sachsen-Anhalt        |2026    |Wahlbezirk     

In [0]:
gold_state_turnout.select(
    "bundesland",
    "wahljahr",
    "wahlkreis_id",
    "gemeinde_id",
    "wahlbezirk_id",
    "gebietstyp"
).distinct().show(100, truncate=False)

+----------+--------+------------+-----------+------------------+----------+
|bundesland|wahljahr|wahlkreis_id|gemeinde_id|wahlbezirk_id     |gebietstyp|
+----------+--------+------------+-----------+------------------+----------+
|Sachsen   |2024    |1           |14523320   |52332000100810    |UB        |
|Sachsen   |2024    |3           |14523190   |523 190 003 00 937|BW        |
|Sachsen   |2024    |5           |14524030   |24030016          |UB        |
|Sachsen   |2024    |9           |145110001  |9204              |UB        |
|Sachsen   |2024    |10          |145110002  |B13A              |BW        |
|Sachsen   |2024    |11          |145110003  |2504              |UB        |
|Sachsen   |2024    |11          |145110003  |6204              |UB        |
|Sachsen   |2024    |16          |14521460   |108               |UB        |
|Sachsen   |2024    |17          |14522180   |211               |UB        |
|Sachsen   |2024    |20          |14522500   |09161953          |BW        |

In [0]:
from pyspark.sql import functions as F
turnout_summary_base = (
    gold_state_turnout
    .filter(F.col("bundesland") != "Bayern")
    .filter(
        ~(
            (F.col("bundesland") == "Nordrhein-Westfalen")
            & (F.col("wahlkreis_id") == "0")
        )
    )
)

In [0]:
state_summary = (
    turnout_summary_base
    .groupBy(
        "bundesland",
        "wahljahr"
    )
    .agg(
        F.sum("wahlberechtigte").alias("wahlberechtigte"),
        F.sum("waehler").alias("waehler"),

        F.sum("gueltige_erststimmen").alias("gueltige_erststimmen"),
        F.sum("ungueltige_erststimmen").alias("ungueltige_erststimmen"),

        F.sum("gueltige_zweitstimmen").alias("gueltige_zweitstimmen"),
        F.sum("ungueltige_zweitstimmen").alias("ungueltige_zweitstimmen")
    )
    .withColumn(
        "wahlbeteiligung_pct",
        F.round(
            F.col("waehler") / F.col("wahlberechtigte") * 100,
            2
        )
    )
)

In [0]:
state_summary.orderBy(
    "bundesland",
    "wahljahr"
).show(truncate=False)

+----------------------+--------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+-------------------+
|bundesland            |wahljahr|wahlberechtigte|waehler|gueltige_erststimmen|ungueltige_erststimmen|gueltige_zweitstimmen|ungueltige_zweitstimmen|wahlbeteiligung_pct|
+----------------------+--------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+-------------------+
|Hamburg               |2020    |1316691        |829497 |NULL                |NULL                  |NULL                 |NULL                   |63.0               |
|Hamburg               |2025    |1313043        |887742 |NULL                |NULL                  |NULL                 |NULL                   |67.61              |
|Mecklenburg-Vorpommern|2021    |1312471        |928807 |910169              |18638                 |913863               |14944                  |70.77        

In [0]:
gold_state_turnout.filter(
    F.col("bundesland") == "Bayern"
).groupBy(
    "gebietstyp"
).agg(
    F.count("*").alias("rows"),
    F.sum("wahlberechtigte").alias("wahlberechtigte"),
    F.sum("waehler").alias("waehler")
).show()

+----------+----+---------------+-------+
|gebietstyp|rows|wahlberechtigte|waehler|
+----------+----+---------------+-------+
|        21| 521|              0| 680598|
|        11| 507|        1820168| 577794|
+----------+----+---------------+-------+



In [0]:
gold_state_turnout.filter(
    F.col("bundesland") == "Bayern"
).orderBy(
    F.desc("wahlberechtigte")
).select(
    "wahlbezirk_id",
    "gebietstyp",
    "wahlberechtigte",
    "waehler"
).show(30, truncate=False)

+-------------+----------+---------------+-------+
|wahlbezirk_id|gebietstyp|wahlberechtigte|waehler|
+-------------+----------+---------------+-------+
|162          |11        |910084         |288897 |
|2210         |11        |2090           |537    |
|105          |11        |2038           |704    |
|106          |11        |2010           |582    |
|1310         |11        |2004           |652    |
|2216         |11        |2003           |560    |
|2402         |11        |2003           |585    |
|715          |11        |1990           |671    |
|1801         |11        |1979           |598    |
|107          |11        |1975           |686    |
|1010         |11        |1972           |527    |
|2201         |11        |1970           |702    |
|2009         |11        |1969           |633    |
|1014         |11        |1969           |577    |
|1103         |11        |1968           |411    |
|2211         |11        |1967           |607    |
|2120         |11        |1967 

In [0]:
state_summary.select(
    "bundesland",
    "wahljahr",
    "wahlberechtigte",
    "waehler",
    "wahlbeteiligung_pct"
).show(truncate=False)

+----------------------+--------+---------------+-------+-------------------+
|bundesland            |wahljahr|wahlberechtigte|waehler|wahlbeteiligung_pct|
+----------------------+--------+---------------+-------+-------------------+
|Sachsen               |2024    |3182683        |2367607|74.39              |
|Sachsen               |2019    |3288643        |2188486|66.55              |
|Sachsen-Anhalt        |2021    |1788930        |1079045|60.32              |
|Sachsen-Anhalt        |2026    |1706851        |1328211|77.82              |
|Mecklenburg-Vorpommern|2021    |1312471        |928807 |70.77              |
|Hamburg               |2025    |1313043        |887742 |67.61              |
|Hamburg               |2020    |1316691        |829497 |63.0               |
|Nordrhein-Westfalen   |2022    |12965858       |7200293|55.53              |
|Nordrhein-Westfalen   |2017    |13164887       |8577221|65.15              |
+----------------------+--------+---------------+-------+-------

In [0]:
bayern_summary_path="abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/gold/bayern_summary_path/"

bayern_summary=spark.read.parquet(bayern_summary_path)

In [0]:
state_summary = state_summary.unionByName(
    bayern_summary,
    allowMissingColumns=True
)

In [0]:
print(state_summary.columns)

['bundesland', 'wahljahr', 'wahlberechtigte', 'waehler', 'gueltige_erststimmen', 'ungueltige_erststimmen', 'gueltige_zweitstimmen', 'ungueltige_zweitstimmen', 'wahlbeteiligung_pct']


In [0]:
state_summary.select("*").where(F.col("bundesland") == "Bayern").show(truncate=False)


+----------+--------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+-------------------+
|bundesland|wahljahr|wahlberechtigte|waehler|gueltige_erststimmen|ungueltige_erststimmen|gueltige_zweitstimmen|ungueltige_zweitstimmen|wahlbeteiligung_pct|
+----------+--------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+-------------------+
|Bayern    |2023    |910084         |629196 |NULL                |NULL                  |NULL                 |NULL                   |69.14              |
+----------+--------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+-------------------+



In [0]:
state_summary.orderBy(
    "bundesland",
    "wahljahr"
).show(truncate=False)

+----------------------+--------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+-------------------+
|bundesland            |wahljahr|wahlberechtigte|waehler|gueltige_erststimmen|ungueltige_erststimmen|gueltige_zweitstimmen|ungueltige_zweitstimmen|wahlbeteiligung_pct|
+----------------------+--------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+-------------------+
|Bayern                |2023    |910084         |629196 |NULL                |NULL                  |NULL                 |NULL                   |69.14              |
|Hamburg               |2020    |1316691        |829497 |NULL                |NULL                  |NULL                 |NULL                   |63.0               |
|Hamburg               |2025    |1313043        |887742 |NULL                |NULL                  |NULL                 |NULL                   |67.61        

In [0]:
state_summary_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/election_turnout/state_summary/"
)

(
    state_summary
    .write
    .mode("overwrite")
    .partitionBy("bundesland")
    .parquet(state_summary_path)
)